# Eco-Travel Advisor — Setup & Demo

**Conversational Agent for Sustainable Tourism Planning (Rasa + NeonDB)**

This notebook is the *setup, data and testing* layer for the project. It does **not** run the
chatbot itself — the Rasa assistant runs as a server and is deployed separately (see below).
What this notebook does, with reproducible outputs for the report:

1. Detect the environment (Google Colab or local Jupyter) and prepare the project.
2. Install dependencies.
3. Load secrets (NeonDB URL, Climatiq key) safely — values are never printed.
4. Validate the curated mock seed data.
5. Test the NeonDB connection.
6. Seed the database and demonstrate idempotent re-runs.
7. *(Later)* Fallback-logic test, Climatiq API test, and demo outputs.

### How to run the actual chatbot
- **Primary:** open the live **HuggingFace Spaces** URL and chat — no setup required.
- **Alternative:** clone the GitHub repository and run `docker compose up` (see README).
- **This notebook:** run the cells top-to-bottom to reproduce the data and test layers.

> The notebook is portable: it works unchanged in Colab and in local Jupyter.

## 1. Environment detection

In [ ]:
import os, sys

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print('Environment:', 'Google Colab' if IN_COLAB else 'local Jupyter')

## 2. Prepare the project

In **Colab** this always fetches a *fresh* clone from GitHub, so every run reflects the latest
push (no stale files). In **local Jupyter** the notebook already sits at the project root, so it
just resolves the path. A quick freshness check prints the number of transport routes loaded.

In [ ]:
REPO_URL = 'https://github.com/yasmiiinay/eco-travel-advisor.git'
PROJECT_DIR = 'eco-travel-advisor'

if IN_COLAB:
    # Always fetch a fresh copy so every run reflects the latest GitHub push.
    if os.path.basename(os.getcwd()) == PROJECT_DIR:
        os.chdir('..')                       # step out of a previous clone
    !rm -rf $PROJECT_DIR
    !git clone $REPO_URL
    if not os.path.isdir(PROJECT_DIR):
        raise FileNotFoundError('git clone failed — check REPO_URL and that the repo is public.')
    os.chdir(PROJECT_DIR)
    PROJECT_ROOT = os.getcwd()
else:
    # Local Jupyter: the notebook already lives at the project root.
    if not (os.path.isdir('actions') and os.path.isdir('data/seed')):
        raise FileNotFoundError('Open this notebook from the project root '
                                '(the folder containing actions/ and data/seed/).')
    PROJECT_ROOT = os.getcwd()

print('Project root:', PROJECT_ROOT)

# Freshness check: how many transport routes were pulled in?
import json
print('transport_option rows:', len(json.load(open('data/seed/transport_option.json'))))

## 3. Install dependencies

In [ ]:
%pip install -q sqlalchemy psycopg2-binary python-dotenv requests

## 4. Load secrets safely

- **Colab:** add `NEON_DATABASE_URL` (and optionally `CLIMATIQ_API_KEY`) in the left-hand
  **Secrets** panel and enable notebook access.
- **Local:** put them in a `.env` file at the project root (never committed).

Only a boolean *configured?* flag is printed — the actual values are never shown.

In [ ]:
def load_secrets():
    if IN_COLAB:
        from google.colab import userdata
        for key in ('NEON_DATABASE_URL', 'CLIMATIQ_API_KEY'):
            try:
                value = userdata.get(key)
                if value:
                    os.environ[key] = value
            except Exception:
                pass  # secret not set / access not granted
    else:
        try:
            from dotenv import load_dotenv
            load_dotenv()
        except ImportError:
            print('python-dotenv not installed; relying on shell environment.')

load_secrets()
print('NEON_DATABASE_URL configured:', bool(os.environ.get('NEON_DATABASE_URL')))
print('CLIMATIQ_API_KEY  configured:', bool(os.environ.get('CLIMATIQ_API_KEY')))

## 5. Validate the curated mock seed data

Checks that every seed file is valid JSON and internally consistent: foreign keys resolve,
sustainability tags exist, every transport mode has an emission factor, and the emissions
figures match `distance x factor`.

In [ ]:
import json, glob

seed = {os.path.basename(p): json.load(open(p, encoding='utf-8'))
        for p in glob.glob('data/seed/*.json')}
print('Seed files loaded:', len(seed))

dest_ids = {d['destination_id'] for d in seed['destination.json']}
tag_names = {t['tag_name'] for t in seed['tags.json']}
factor_modes = {e['mode'] for e in seed['emission_factor.json']}
factor = {e['mode']: e['kg_co2e_per_passenger_km'] for e in seed['emission_factor.json']}
errors = []

for fn in ('hotel.json', 'experience.json', 'transport_option.json', 'offset_option.json'):
    for row in seed[fn]:
        if row['destination_id'] not in dest_ids:
            errors.append(f"{fn}: bad destination_id {row['destination_id']}")
        for tg in row.get('sustainability_tags', []):
            if tg not in tag_names:
                errors.append(f"{fn}: unknown tag '{tg}'")

for row in seed['transport_option.json']:
    if row['mode'] not in factor_modes:
        errors.append(f"transport_option.json: mode '{row['mode']}' has no emission factor")
    expected = round(row['estimated_distance_km'] * factor[row['mode']], 1)
    if abs(expected - row['estimated_emissions_kg_per_person']) > 0.2:
        errors.append(f"option {row['option_id']}: emissions mismatch")

for d in sorted(dest_ids):
    h = sum(1 for x in seed['hotel.json'] if x['destination_id'] == d)
    e = sum(1 for x in seed['experience.json'] if x['destination_id'] == d)
    print(f'  destination {d}: {h} hotels, {e} experiences')

print('\nVALIDATION:', 'PASSED' if not errors else f'{len(errors)} ERROR(S): {errors}')

## 6. Test the NeonDB connection

Imports the ORM models from `actions/db.py` and runs a trivial `SELECT 1`. Connection
details are never displayed. If the database is unreachable, the project still works on the
local JSON fallback (demonstrated later).

In [ ]:
sys.path.insert(0, os.path.join(PROJECT_ROOT, 'actions'))
import db

print('DB configured:', db.is_db_configured())
if db.is_db_configured():
    from sqlalchemy import text
    try:
        with db.get_engine().connect() as conn:
            conn.execute(text('SELECT 1'))
        print('NeonDB connection: OK')
    except Exception as exc:
        print('NeonDB connection FAILED:', type(exc).__name__)
else:
    print('NEON_DATABASE_URL not set — skipping (JSON fallback will be used).')

## 7. Seed the database

Runs `actions/seed_db.py`, which creates the tables and idempotently upserts every seed
file. On the **first** run every table reports `inserted`.

In [ ]:
import subprocess

def run_seed():
    result = subprocess.run([sys.executable, 'actions/seed_db.py'],
                            capture_output=True, text=True)
    print(result.stdout)
    if result.returncode != 0:
        print('--- stderr ---')
        print(result.stderr)
    return result.returncode

run_seed()

## 8. Demonstrate idempotency

Re-running the seeder must **not** create duplicates: every table should now report
`updated` with `inserted = 0`. This screenshot is useful evidence in the testing section.

In [ ]:
run_seed()

## 9. Fallback-logic test

Demonstrates the cascade **NeonDB -> local JSON**. The same query is run twice: first with
the database configured (`source: neondb`), then with the database forced off to prove the
chatbot degrades gracefully to the local JSON seed files (`source: json_fallback`).

This is direct evidence of the resilience requirement for the testing section.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(PROJECT_ROOT, 'actions'))
import repository as repo

# --- Tier 1: NeonDB (the connection string was loaded in the secrets cell) ---
print('NEON_DATABASE_URL configured:', bool(os.environ.get('NEON_DATABASE_URL')))
dest, src1 = repo.resolve_destination('Pariiis')          # typo-tolerant
print(f"resolve_destination('Pariiis') -> {dest['city']}  | source: {src1}")
opts, src1b = repo.get_transport_options('London', dest['destination_id'])
print(f"greenest transport: {opts[0]['mode']} ~{opts[0]['estimated_emissions_kg_per_person']} kg  | source: {src1b}")

# --- Tier 2: force NeonDB OFF to prove graceful fallback to local JSON ---
saved = os.environ.pop('NEON_DATABASE_URL', None)
repo._seed_cache.clear()
dest2, src2 = repo.resolve_destination('Berln')           # another typo
print(f"\n[DB forced OFF] resolve_destination('Berln') -> {dest2['city']}  | source: {src2}")
opts2, src2b = repo.get_transport_options('Madrid', dest2['destination_id'])
print(f"[DB forced OFF] transport options for Madrid->{dest2['city']}: {len(opts2)} found  | source: {src2b}")
if saved:
    os.environ['NEON_DATABASE_URL'] = saved               # restore for later cells

print('\nFallback cascade verified:', src1 == 'neondb' and src2 == 'json_fallback')

## 10. Climatiq API test  *(enabled after `actions/carbon.py`)*

Will call Climatiq for a sample route, show the live carbon estimate, then disable the key
to show the seamless fallback to the stored emission factors.

## 11. Demo outputs for the report  *(to be added)*

Worked end-to-end examples (e.g. *London -> Copenhagen, 2 travellers, lowest-carbon
preference*) with the carbon estimate, colour-coded recommendations and offset suggestion,
captured as figures for the final report.

## 12. Rasa setup (isolated Python 3.10 environment)

Rasa 3.6.x needs **Python 3.10**, while this Colab kernel is 3.12. So we build a separate
3.10 virtual environment just for Rasa and call it through its own binary. The notebook kernel
stays 3.12 (the data cells above keep working); the two environments sit side by side.

The first install pulls in TensorFlow and other heavy dependencies, so it takes a few minutes.

In [ ]:
# Paths to the Rasa environment (Colab) or the active local 3.10 venv.
RASA  = '/content/rasa-venv/bin/rasa'   if IN_COLAB else 'rasa'
PYBIN = '/content/rasa-venv/bin/python' if IN_COLAB else 'python'
print('Using rasa at:', RASA)

In [ ]:
# One-time: create the Python 3.10 venv and install the pinned requirements.
if IN_COLAB:
    !apt-get -qq install -y python3.10-venv python3.10-distutils >/dev/null
    !python3.10 -m venv /content/rasa-venv
    !/content/rasa-venv/bin/pip install -q --upgrade pip
    !/content/rasa-venv/bin/pip install -q -r requirements.txt
    !/content/rasa-venv/bin/rasa --version
else:
    print('Local: activate a Python 3.10 venv, then run  pip install -r requirements.txt')

## 13. Validate the project (imports + data consistency)

First confirm the custom actions import cleanly (rasa_sdk + repository + carbon), then run
Rasa's own validator over the domain, NLU, rules and stories.

In [ ]:
# Custom actions import test (no database connection is opened).
!{PYBIN} -c "import actions.actions; print('actions.py imports OK')"

In [ ]:
# Validate domain / nlu / rules / stories for consistency.
!{RASA} data validate

## 14. Train the assistant

Trains the NLU model and the dialogue policies defined in `config.yml`. The trained model is
written to `models/` (git-ignored). This takes a couple of minutes on CPU.

In [ ]:
!{RASA} train

## 15. Test the model (NLU + dialogue)

Evaluates the trained model. Reports and a confusion matrix are written to `results/` -
useful figures for the testing section of the report.

In [ ]:
!MPLBACKEND=Agg {RASA} test nlu -u data/nlu.yml
!MPLBACKEND=Agg {RASA} test core --stories data/stories.yml
!ls -R results 2>/dev/null || echo 'Results are written to the results/ folder.'

## 16. Try the assistant

`rasa shell` is interactive, so it is best run in a local terminal:
```
rasa run actions     # terminal 1 (custom actions, port 5055)
rasa shell           # terminal 2 (chat with the bot)
```
For a live web demo (the responsive UI in `frontend/`), the assistant is deployed on
HuggingFace Spaces in the deployment step.